## קבוצות ליגת העל ומספר תושבים בעיר

במחברת הזו:
1. מחלצים את כל הקבוצות השונות שהופיעו בליגת העל לאורך כל העונות בקובץ המשחקים המאוחד.
2. בונים טבלת מיפוי `קבוצה -> עיר`.
3. מושכים את מספר התושבים של ערי הקבוצות מתוך נתוני הלמ"ס לשנת 2024.

הפלט יישמר כקבצי CSV לשימוש בהמשך הניתוח.

In [1]:
from __future__ import annotations

import json
import shutil
import subprocess
from pathlib import Path

import pandas as pd
from IPython.display import display


def find_project_root(start: Path | None = None) -> Path:
    current = (start or Path.cwd()).resolve()
    for candidate in [current, *current.parents]:
        if (candidate / 'notebooks' / 'data' / 'matches').exists():
            return candidate
    return current


ROOT = find_project_root()
MATCHES_FILE = ROOT / 'notebooks' / 'data' / 'matches' / 'matches_all_seasons_ligat_haal_transfermarkt.csv'
OUTPUT_DIR = ROOT / 'notebooks' / 'data' / 'demographic'
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

POWERSHELL_EXE = shutil.which('powershell') or shutil.which('pwsh')
if not POWERSHELL_EXE:
    raise RuntimeError('PowerShell was not found.')

print(f'ROOT: {ROOT}')
print(f'MATCHES_FILE: {MATCHES_FILE}')
print(f'OUTPUT_DIR: {OUTPUT_DIR}')

ROOT: C:\Users\nitib\dev-lab\ligat_haal_project\ligat_haal_project
MATCHES_FILE: C:\Users\nitib\dev-lab\ligat_haal_project\ligat_haal_project\notebooks\data\matches\matches_all_seasons_ligat_haal_transfermarkt.csv
OUTPUT_DIR: C:\Users\nitib\dev-lab\ligat_haal_project\ligat_haal_project\notebooks\data\demographic


In [2]:
matches_df = pd.read_csv(MATCHES_FILE)
unique_teams = sorted(set(matches_df['home']).union(set(matches_df['away'])))

team_city_mapping = [
    {'team_raw': 'B. Jerusalem', 'club_name': 'Beitar Jerusalem', 'city_he': 'ירושלים', 'city_en': 'Jerusalem'},
    {'team_raw': 'Bnei Sakhnin', 'club_name': 'Bnei Sakhnin', 'city_he': "סח'נין", 'city_en': 'Sakhnin'},
    {'team_raw': 'Bnei Yehuda', 'club_name': 'Bnei Yehuda Tel Aviv', 'city_he': 'תל אביב -יפו', 'city_en': 'Tel Aviv-Yafo'},
    {'team_raw': 'FC Ashdod', 'club_name': 'FC Ashdod', 'city_he': 'אשדוד', 'city_en': 'Ashdod'},
    {'team_raw': 'H Rishon leZion', 'club_name': 'Hapoel Rishon LeZion', 'city_he': 'ראשון לציון', 'city_en': 'Rishon LeZion'},
    {'team_raw': 'H. Ashkelon', 'club_name': 'Hapoel Ashkelon', 'city_he': 'אשקלון', 'city_en': 'Ashkelon'},
    {'team_raw': 'H. Beer Sheva', 'club_name': 'Hapoel Beer Sheva', 'city_he': 'באר שבע', 'city_en': 'Beer Sheva'},
    {'team_raw': 'H. Jerusalem', 'club_name': 'Hapoel Jerusalem', 'city_he': 'ירושלים', 'city_en': 'Jerusalem'},
    {'team_raw': 'H. Kfar Saba', 'club_name': 'Hapoel Kfar Saba', 'city_he': 'כפר סבא', 'city_en': 'Kfar Saba'},
    {'team_raw': 'H. Nof HaGalil', 'club_name': 'Hapoel Nof HaGalil', 'city_he': 'נוף הגליל', 'city_en': 'Nof HaGalil'},
    {'team_raw': 'H. Petah Tikva', 'club_name': 'Hapoel Petah Tikva', 'city_he': 'פתח תקווה', 'city_en': 'Petah Tikva'},
    {'team_raw': 'H. Ramat Gan', 'club_name': 'Hapoel Ramat Gan', 'city_he': 'רמת גן', 'city_en': 'Ramat Gan'},
    {'team_raw': 'Hakoah Amidar', 'club_name': 'Hakoah Amidar Ramat Gan', 'city_he': 'רמת גן', 'city_en': 'Ramat Gan'},
    {'team_raw': 'Hapoel Acre', 'club_name': 'Hapoel Acre', 'city_he': 'עכו', 'city_en': 'Acre'},
    {'team_raw': 'Hapoel Hadera', 'club_name': 'Hapoel Hadera', 'city_he': 'חדרה', 'city_en': 'Hadera'},
    {'team_raw': 'Hapoel Haifa', 'club_name': 'Hapoel Haifa', 'city_he': 'חיפה', 'city_en': 'Haifa'},
    {'team_raw': 'Hapoel Raanana', 'club_name': 'Hapoel Raanana', 'city_he': 'רעננה', 'city_en': 'Raanana'},
    {'team_raw': 'Hapoel Tel Aviv', 'club_name': 'Hapoel Tel Aviv', 'city_he': 'תל אביב -יפו', 'city_en': 'Tel Aviv-Yafo'},
    {'team_raw': 'Ironi Tiberias', 'club_name': 'Ironi Tiberias', 'city_he': 'טבריה', 'city_en': 'Tiberias'},
    {'team_raw': 'Kiryat Shmona', 'club_name': 'Ironi Kiryat Shmona', 'city_he': 'קריית שמונה', 'city_en': 'Kiryat Shmona'},
    {'team_raw': 'M. Ahi Nazareth', 'club_name': 'Maccabi Ahi Nazareth', 'city_he': 'נצרת', 'city_en': 'Nazareth'},
    {'team_raw': 'M. Bnei Reineh', 'club_name': 'Maccabi Bnei Reineh', 'city_he': 'ריינה', 'city_en': 'Reineh'},
    {'team_raw': 'M. Petah Tikva', 'club_name': 'Maccabi Petah Tikva', 'city_he': 'פתח תקווה', 'city_en': 'Petah Tikva'},
    {'team_raw': 'M. Tel Aviv', 'club_name': 'Maccabi Tel Aviv', 'city_he': 'תל אביב -יפו', 'city_en': 'Tel Aviv-Yafo'},
    {'team_raw': 'Maccabi Haifa', 'club_name': 'Maccabi Haifa', 'city_he': 'חיפה', 'city_en': 'Haifa'},
    {'team_raw': 'Maccabi Herzlya', 'club_name': 'Maccabi Herzliya', 'city_he': 'הרצלייה', 'city_en': 'Herzliya'},
    {'team_raw': 'Maccabi Netanya', 'club_name': 'Maccabi Netanya', 'city_he': 'נתניה', 'city_en': 'Netanya'},
    {'team_raw': 'Ness Ziona', 'club_name': 'Sektzia Ness Ziona', 'city_he': 'נס ציונה', 'city_en': 'Ness Ziona'},
    {'team_raw': 'Ramat haSharon', 'club_name': 'Hapoel Nir Ramat HaSharon', 'city_he': 'רמת השרון', 'city_en': 'Ramat HaSharon'},
]

teams_df = pd.DataFrame(team_city_mapping).sort_values(['city_he', 'club_name']).reset_index(drop=True)
missing_mapping = sorted(set(unique_teams) - set(teams_df['team_raw']))
extra_mapping = sorted(set(teams_df['team_raw']) - set(unique_teams))

if missing_mapping:
    raise ValueError(f'Missing team mappings: {missing_mapping}')
if extra_mapping:
    raise ValueError(f'Extra mapped teams not found in matches file: {extra_mapping}')

teams_df.to_csv(OUTPUT_DIR / 'ligat_haal_unique_teams_and_cities.csv', index=False, encoding='utf-8-sig')

print(f'Unique team names found: {len(unique_teams)}')
print(f'Mapped teams: {len(teams_df)}')
display(teams_df)

Unique team names found: 29
Mapped teams: 29


,team_raw,club_name,city_he,city_en
0,FC Ashdod,FC Ashdod,אשדוד,Ashdod
1,H. Ashkelon,Hapoel Ashkelon,אשקלון,Ashkelon
2,H. Beer Sheva,Hapoel Beer Sheva,באר שבע,Beer Sheva
3,Maccabi Herzlya,Maccabi Herzliya,הרצלייה,Herzliya
4,Hapoel Hadera,Hapoel Hadera,חדרה,Hadera
5,Hapoel Haifa,Hapoel Haifa,חיפה,Haifa
6,Maccabi Haifa,Maccabi Haifa,חיפה,Haifa
7,Ironi Tiberias,Ironi Tiberias,טבריה,Tiberias
8,B. Jerusalem,Beitar Jerusalem,ירושלים,Jerusalem
9,H. Jerusalem,Hapoel Jerusalem,ירושלים,Jerusalem


In [3]:
BASE_URL = 'https://boardsgenerator.cbs.gov.il/Handlers/WebParts/YishuvimHandler.ashx'


def fetch_base_data_year(year: int = 2024, timeout_seconds: int = 240) -> pd.DataFrame:
    powershell_script = """
[Console]::OutputEncoding = [System.Text.Encoding]::UTF8
$ProgressPreference = 'SilentlyContinue'
$ErrorActionPreference = 'Stop'
$year = __YEAR__

function Get-Page([int]$pageNumber) {
    $filtersJson = @{ Years = $year } | ConvertTo-Json -Compress
    $query = @{
        mode = 'GridData'
        dataMode = 'Yeshuv'
        subject = 'BaseData'
        filters = $filtersJson
        filtersearch = ''
        pageNumber = [string]$pageNumber
        search = ''
        language = 'Hebrew'
    }
    $queryString = ($query.GetEnumerator() | ForEach-Object {
        '{0}={1}' -f [uri]::EscapeDataString([string]$_.Key), [uri]::EscapeDataString([string]$_.Value)
    }) -join '&'
    $url = '__BASE_URL__?' + $queryString
    $response = Invoke-WebRequest -UseBasicParsing -Uri $url -TimeoutSec 90 -ErrorAction Stop
    return ($response.Content | ConvertFrom-Json -ErrorAction Stop)
}

$firstPage = Get-Page -pageNumber 1
$rows = New-Object System.Collections.ArrayList
foreach ($row in @($firstPage.Table)) { [void]$rows.Add($row) }

for ($page = 2; $page -le [int]$firstPage.TotalPages; $page++) {
    $pageData = Get-Page -pageNumber $page
    foreach ($row in @($pageData.Table)) { [void]$rows.Add($row) }
}

$rows | ConvertTo-Json -Depth 6 -Compress
"""
    powershell_script = (
        powershell_script
        .replace('__BASE_URL__', BASE_URL)
        .replace('__YEAR__', str(year))
    )

    result = subprocess.run(
        [POWERSHELL_EXE, '-NoProfile', '-Command', powershell_script],
        capture_output=True,
        text=True,
        encoding='utf-8',
        errors='replace',
        timeout=timeout_seconds,
        check=False,
    )
    if result.returncode != 0:
        raise RuntimeError(result.stderr.strip() or f'PowerShell exited with code {result.returncode}')

    content = result.stdout.strip()
    if not content:
        raise RuntimeError('Empty response body from CBS BaseData endpoint')

    payload = json.loads(content)
    if isinstance(payload, dict):
        rows = [payload]
    else:
        rows = payload

    city_df = pd.DataFrame(rows).rename(columns={
        'SemelYishuv': 'settlement_code',
        'Name': 'city_he',
        'Machoz': 'district_he',
        'Nafa': 'subdistrict_he',
        'TzuratYishuv': 'settlement_form_he',
        'MaamadMuni': 'municipal_status_he',
        'EzorTivai': 'natural_region_he',
        'PepoleNumber': 'population_total',
        'PepoleNumberJewishWithOther': 'population_jewish_and_other',
        'PepoleNumberJewish': 'population_jewish',
        'PepoleNumberArab': 'population_arab',
        'SemelEshkolRashuyotMekomiyot': 'cluster_he',
    })

    for col in ['population_total', 'population_jewish_and_other', 'population_jewish', 'population_arab']:
        if col in city_df.columns:
            city_df[col] = pd.to_numeric(
                city_df[col].astype(str).str.replace(',', '', regex=False).replace({'-': pd.NA, 'nan': pd.NA}),
                errors='coerce',
            )

    city_df['year'] = year
    return city_df


city_population_year = 2024
city_base_df = fetch_base_data_year(city_population_year)
team_city_population_df = teams_df.merge(
    city_base_df[['city_he', 'settlement_code', 'population_total', 'district_he', 'subdistrict_he', 'municipal_status_he', 'year']],
    on='city_he',
    how='left',
)

missing_cities = team_city_population_df[team_city_population_df['population_total'].isna()]['city_he'].drop_duplicates().tolist()
if missing_cities:
    print('Cities not matched in CBS BaseData:')
    print(missing_cities)

team_city_population_df.to_csv(OUTPUT_DIR / 'ligat_haal_teams_city_population_2024.csv', index=False, encoding='utf-8-sig')

print(f'CBS city records loaded: {len(city_base_df):,}')
print(f'Teams with city population table: {len(team_city_population_df)}')
display(team_city_population_df.sort_values(['city_he', 'club_name']).reset_index(drop=True))

CBS city records loaded: 1,291
Teams with city population table: 29


,team_raw,club_name,city_he,city_en,settlement_code,population_total,district_he,subdistrict_he,municipal_status_he,year
0,FC Ashdod,FC Ashdod,אשדוד,Ashdod,70,228562.0,הדרום,אשקלון,עירייה,2024
1,H. Ashkelon,Hapoel Ashkelon,אשקלון,Ashkelon,7100,166865.0,הדרום,אשקלון,עירייה,2024
2,H. Beer Sheva,Hapoel Beer Sheva,באר שבע,Beer Sheva,9000,223587.0,הדרום,באר שבע,עירייה,2024
3,Maccabi Herzlya,Maccabi Herzliya,הרצלייה,Herzliya,6400,110885.0,תל אביב,תל אביב,עירייה,2024
4,Hapoel Hadera,Hapoel Hadera,חדרה,Hadera,6500,107974.0,חיפה,חדרה,עירייה,2024
5,Hapoel Haifa,Hapoel Haifa,חיפה,Haifa,4000,297083.0,חיפה,חיפה,עירייה,2024
6,Maccabi Haifa,Maccabi Haifa,חיפה,Haifa,4000,297083.0,חיפה,חיפה,עירייה,2024
7,Ironi Tiberias,Ironi Tiberias,טבריה,Tiberias,6700,52122.0,הצפון,כנרת,עירייה,2024
8,B. Jerusalem,Beitar Jerusalem,ירושלים,Jerusalem,3000,1050151.0,ירושלים,ירושלים,עירייה,2024
9,H. Jerusalem,Hapoel Jerusalem,ירושלים,Jerusalem,3000,1050151.0,ירושלים,ירושלים,עירייה,2024


In [7]:
import time


def fetch_base_data_year(year: int = 2024, timeout_seconds: int = 240, max_attempts: int = 6) -> pd.DataFrame:
    powershell_script = """
[Console]::OutputEncoding = [System.Text.Encoding]::UTF8
$ProgressPreference = 'SilentlyContinue'
$ErrorActionPreference = 'Stop'
$year = __YEAR__
$headers = @{
    'User-Agent' = 'Mozilla/5.0'
    'Referer' = 'https://boardsgenerator.cbs.gov.il/pages/WebParts/YishuvimPage.aspx?mode=Yishuv'
    'Accept' = 'application/json, text/plain, */*'
}

function Get-Page([int]$pageNumber) {
    $filtersJson = @{ Years = $year } | ConvertTo-Json -Compress
    $query = @{
        mode = 'GridData'
        dataMode = 'Yeshuv'
        subject = 'BaseData'
        filters = $filtersJson
        filtersearch = ''
        pageNumber = [string]$pageNumber
        search = ''
        language = 'Hebrew'
    }
    $queryString = ($query.GetEnumerator() | ForEach-Object {
        '{0}={1}' -f [uri]::EscapeDataString([string]$_.Key), [uri]::EscapeDataString([string]$_.Value)
    }) -join '&'
    $url = '__BASE_URL__?' + $queryString
    $response = Invoke-WebRequest -UseBasicParsing -Headers $headers -Uri $url -TimeoutSec 90 -ErrorAction Stop
    if (-not $response.Content) {
        throw 'Empty response body'
    }
    if ($response.Content -like '<html*Request Rejected*' -or $response.Content -like '*Request Rejected*') {
        throw 'Request Rejected by CBS server'
    }
    return ($response.Content | ConvertFrom-Json -ErrorAction Stop)
}

$firstPage = Get-Page -pageNumber 1
$rows = New-Object System.Collections.ArrayList
foreach ($row in @($firstPage.Table)) { [void]$rows.Add($row) }
Start-Sleep -Milliseconds 250

for ($page = 2; $page -le [int]$firstPage.TotalPages; $page++) {
    $pageData = Get-Page -pageNumber $page
    foreach ($row in @($pageData.Table)) { [void]$rows.Add($row) }
    Start-Sleep -Milliseconds 250
}

$rows | ConvertTo-Json -Depth 6 -Compress
"""
    powershell_script = (
        powershell_script
        .replace('__BASE_URL__', BASE_URL)
        .replace('__YEAR__', str(year))
    )

    last_error = None
    for attempt in range(1, max_attempts + 1):
        result = subprocess.run(
            [POWERSHELL_EXE, '-NoProfile', '-Command', powershell_script],
            capture_output=True,
            text=True,
            encoding='utf-8',
            errors='replace',
            timeout=timeout_seconds,
            check=False,
        )

        if result.returncode == 0:
            content = result.stdout.strip()
            if content and 'Request Rejected' not in content and not content.lower().startswith('<html'):
                try:
                    payload = json.loads(content)
                    rows = [payload] if isinstance(payload, dict) else payload
                    city_df = pd.DataFrame(rows).rename(columns={
                        'SemelYishuv': 'settlement_code',
                        'Name': 'city_he',
                        'Machoz': 'district_he',
                        'Nafa': 'subdistrict_he',
                        'TzuratYishuv': 'settlement_form_he',
                        'MaamadMuni': 'municipal_status_he',
                        'EzorTivai': 'natural_region_he',
                        'PepoleNumber': 'population_total',
                        'PepoleNumberJewishWithOther': 'population_jewish_and_other',
                        'PepoleNumberJewish': 'population_jewish',
                        'PepoleNumberArab': 'population_arab',
                        'SemelEshkolRashuyotMekomiyot': 'cluster_he',
                    })
                    for col in ['population_total', 'population_jewish_and_other', 'population_jewish', 'population_arab']:
                        if col in city_df.columns:
                            city_df[col] = pd.to_numeric(
                                city_df[col].astype(str).str.replace(',', '', regex=False).replace({'-': pd.NA, 'nan': pd.NA}),
                                errors='coerce',
                            )
                    city_df['year'] = year
                    return city_df
                except Exception as exc:
                    last_error = exc
            else:
                last_error = RuntimeError(result.stderr.strip() or content[:300] or 'Unknown empty/HTML response')
        else:
            last_error = RuntimeError(result.stderr.strip() or f'PowerShell exited with code {result.returncode}')

        wait_seconds = min(30, attempt * 4)
        print(f'Retry {attempt}/{max_attempts} for CBS BaseData {year} after error: {last_error}')
        time.sleep(wait_seconds)

    raise RuntimeError(f'Failed to fetch CBS BaseData for {year}') from last_error

## טבלה עונתית לכל הקבוצות

התא הבא בונה טבלה לכל עונה בליגת העל, אבל משתמש בטבלת ייחוס דמוגרפית אחת במקום לנסות למשוך נתוני למ"ס לכל שנה מחדש.

מה תקבלו:
- אילו קבוצות השתתפו בכל עונה
- לאיזו עיר כל קבוצה משויכת
- מספר התושבים של עיר הקבוצה לפי שנת הייחוס הזמינה במחברת
- שנת הייחוס עצמה, כדי שיהיה ברור שזהו מדד קבוע להשוואה בין קבוצות ועונות

זה מתאים למקרה שבו רוצים מדד גודל שוק יציב יחסית, בלי להיתקע על חסימות מהאתר של הלמ"ס.

In [4]:
season_team_presence = (
    pd.concat(
        [
            matches_df[['season', 'season_year', 'home']].rename(columns={'home': 'team_raw'}),
            matches_df[['season', 'season_year', 'away']].rename(columns={'away': 'team_raw'}),
        ],
        ignore_index=True,
    )
    .drop_duplicates()
    .sort_values(['season_year', 'season', 'team_raw'])
    .reset_index(drop=True)
)

reference_columns = [
    'team_raw',
    'club_name',
    'city_he',
    'city_en',
    'settlement_code',
    'population_total',
    'district_he',
    'subdistrict_he',
    'municipal_status_he',
    'year',
]

reference_population_df = team_city_population_df[reference_columns].rename(
    columns={'year': 'population_reference_year'}
)

season_team_population_df = (
    season_team_presence
    .merge(reference_population_df, on='team_raw', how='left')
    .assign(population_year=lambda df: df['population_reference_year'])
    .assign(population_year_used=lambda df: df['population_reference_year'])
    [[
        'season',
        'season_year',
        'population_reference_year',
        'population_year',
        'population_year_used',
        'team_raw',
        'club_name',
        'city_he',
        'city_en',
        'settlement_code',
        'population_total',
        'district_he',
        'subdistrict_he',
        'municipal_status_he',
    ]]
    .sort_values(['season_year', 'city_he', 'club_name'])
    .reset_index(drop=True)
)

season_team_population_df.to_csv(
    OUTPUT_DIR / 'ligat_haal_teams_city_population_by_season.csv',
    index=False,
    encoding='utf-8-sig',
)

season_summary_df = (
    season_team_population_df.groupby(['season', 'season_year', 'population_reference_year'], as_index=False)
    .agg(
        teams=('club_name', 'count'),
        cities=('city_he', 'nunique'),
        matched_population=('population_total', lambda s: int(s.notna().sum())),
        total_city_population=('population_total', 'sum'),
    )
)

season_summary_df.to_csv(
    OUTPUT_DIR / 'ligat_haal_teams_city_population_by_season_summary.csv',
    index=False,
    encoding='utf-8-sig',
)

print(f'Per-season rows created: {len(season_team_population_df):,}')
print(f'Seasons covered: {season_team_population_df["season"].nunique()}')
print(f'Reference population year used for all rows: {int(season_team_population_df["population_reference_year"].dropna().iloc[0])}')
print('Saved detailed table to:')
print(OUTPUT_DIR / 'ligat_haal_teams_city_population_by_season.csv')
print('Saved season summary to:')
print(OUTPUT_DIR / 'ligat_haal_teams_city_population_by_season_summary.csv')

display(season_summary_df)
display(season_team_population_df.head(30))

Per-season rows created: 280
Seasons covered: 20
Reference population year used for all rows: 2024
Saved detailed table to:
C:\Users\nitib\dev-lab\ligat_haal_project\ligat_haal_project\notebooks\data\demographic\ligat_haal_teams_city_population_by_season.csv
Saved season summary to:
C:\Users\nitib\dev-lab\ligat_haal_project\ligat_haal_project\notebooks\data\demographic\ligat_haal_teams_city_population_by_season_summary.csv


,season,season_year,population_reference_year,teams,cities,matched_population,total_city_population
0,2006/07,2006,2024,12,9,12,4218651.0
1,2007/08,2007,2024,12,10,12,3834169.0
2,2008/09,2008,2024,12,9,12,4066519.0
3,2009/10,2009,2024,16,12,16,4774303.0
4,2010/11,2010,2024,16,12,16,4806938.0
5,2011/12,2011,2024,16,12,16,4775966.0
6,2012/13,2012,2024,14,11,14,4148127.0
7,2013/14,2013,2024,14,11,14,4094439.0
8,2014/15,2014,2024,14,11,14,4055894.0
9,2015/16,2015,2024,14,11,14,4151239.0


,season,season_year,population_reference_year,population_year,population_year_used,team_raw,club_name,city_he,city_en,settlement_code,population_total,district_he,subdistrict_he,municipal_status_he
0,2006/07,2006,2024,2024,2024,FC Ashdod,FC Ashdod,אשדוד,Ashdod,70,228562.0,הדרום,אשקלון,עירייה
1,2006/07,2006,2024,2024,2024,Maccabi Herzlya,Maccabi Herzliya,הרצלייה,Herzliya,6400,110885.0,תל אביב,תל אביב,עירייה
2,2006/07,2006,2024,2024,2024,Maccabi Haifa,Maccabi Haifa,חיפה,Haifa,4000,297083.0,חיפה,חיפה,עירייה
3,2006/07,2006,2024,2024,2024,B. Jerusalem,Beitar Jerusalem,ירושלים,Jerusalem,3000,1050151.0,ירושלים,ירושלים,עירייה
4,2006/07,2006,2024,2024,2024,H. Kfar Saba,Hapoel Kfar Saba,כפר סבא,Kfar Saba,6900,99410.0,המרכז,פתח תקווה,עירייה
5,2006/07,2006,2024,2024,2024,Maccabi Netanya,Maccabi Netanya,נתניה,Netanya,7400,234812.0,המרכז,השרון,עירייה
6,2006/07,2006,2024,2024,2024,H. Petah Tikva,Hapoel Petah Tikva,פתח תקווה,Petah Tikva,7900,270403.0,המרכז,פתח תקווה,עירייה
7,2006/07,2006,2024,2024,2024,M. Petah Tikva,Maccabi Petah Tikva,פתח תקווה,Petah Tikva,7900,270403.0,המרכז,פתח תקווה,עירייה
8,2006/07,2006,2024,2024,2024,Hakoah Amidar,Hakoah Amidar Ramat Gan,רמת גן,Ramat Gan,8600,172242.0,תל אביב,,עירייה
9,2006/07,2006,2024,2024,2024,Bnei Yehuda,Bnei Yehuda Tel Aviv,תל אביב -יפו,Tel Aviv-Yafo,5000,494900.0,תל אביב,תל אביב,עירייה
